# 주간 아파트 매매/전세 가격지수 실습 (Step-by-Step)
이 노트북은 **초보자 실습용**입니다. 각 단계에서 **셀 실행 → 결과 확인 → 다음 단계로 진행**하세요.

## 실습 목표
1) KB 주간 아파트 **매매/전세 가격지수** 엑셀을 불러온다.
2) 서울 **25개 구**만 남기고(집계 지역 제외) `wide(날짜×구)` 형태로 만든다.
3) `long(날짜 index, 지역명+지수)` 형태로 바꾼다.
4) 강남구 시계열/상관관계/격차 Top 구/연도별 집계를 시각화한다.

⚠️ **중요**: 셀을 건너뛰어 실행하면(순서 꼬임) 변수 미정의 에러가 날 수 있어요. 위에서부터 차례대로 실행하세요.

## 0. 라이브러리 불러오기 & 폰트 설정
- Windows(로컬 Jupyter): `Malgun Gothic` 권장
- Colab(리눅스): `NanumGothic` 설치 후 사용 가능

아래 셀을 실행한 뒤, 경고 없이 한글이 보이는지 확인하세요.

In [1]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ✅ Windows 로컬(Jupyter)에서는 보통 'Malgun Gothic'이 안전합니다.
# (Colab이라면 NanumGothic 설치 후 'NanumGothic'로 바꿔도 됩니다.)
plt.rcParams["font.family"] = "Malgun Gothic"
mpl.rcParams["axes.unicode_minus"] = False

print("현재 작업 폴더:", os.getcwd())
print("pandas 버전:", pd.__version__)

현재 작업 폴더: c:\Users\Admin\hipython
pandas 버전: 2.3.3


## 1. 데이터 불러오기
실습 폴더 구조(권장)
```
hipython/
 ├─ 45_apt_price_data_analysis__practice.ipynb
 └─ data/
     ├─ 주간 아파트 매매가격지수_20260210.xlsx
     └─ 주간 아파트 전세가격지수_20260210.xlsx
```

아래 셀을 실행해서 `파일 존재(True)`가 나오고, `shape`가 출력되는지 확인하세요.

In [2]:
from pathlib import Path

SALE_PATH = Path("./data/주간 아파트 매매가격지수_20260210.xlsx")
RENT_PATH = Path("./data/주간 아파트 전세가격지수_20260210.xlsx")

print("매매 파일 존재:", SALE_PATH.exists(), "|", SALE_PATH.resolve())
print("전세 파일 존재:", RENT_PATH.exists(), "|", RENT_PATH.resolve())

df_sale = pd.read_excel(SALE_PATH)
df_rent = pd.read_excel(RENT_PATH)

df_sale.shape, df_rent.shape

매매 파일 존재: True | C:\Users\Admin\hipython\data\주간 아파트 매매가격지수_20260210.xlsx
전세 파일 존재: True | C:\Users\Admin\hipython\data\주간 아파트 전세가격지수_20260210.xlsx


((282, 899), (282, 899))

## 2. 원본 데이터 빠르게 확인
여기서는 **컬럼 이름/지역명 형태/날짜 컬럼 형태**를 확인합니다.

아래 셀 실행 후:
- `지역명`이 실제로 어떤 문자열 형태인지(예: `강남구` vs `서울 강남구`) 확인
- 날짜가 컬럼으로 들어있는지 확인


In [ ]:
display(df_sale.head(3))
print("컬럼 예시(앞 10개):", df_sale.columns[:10].tolist())
print("지역명 예시(앞 20개):", df_sale["지역명"].astype(str).head(20).tolist())
print("'서울' 포함 행 수:", df_sale["지역명"].astype(str).str.contains("서울", na=False).sum())

## 3. 서울 25개 구만 남기기 (wide 형태)
목표:
- `df_sale_seoul`, `df_rent_seoul` 생성
- index = 날짜, columns = 서울 25개 구
- 기대 shape: **(831, 25)** (데이터 버전에 따라 831이 약간 달라도 됩니다)

### 실습 포인트
1) `melt`로 long 변환
2) `서울 25개구` 리스트로 필터(가장 단순 & 실습 친화)
3) pivot으로 wide 만들기
4) 중복 컬럼 제거(실습 중 자주 등장)


In [ ]:
SEOUL_GU_25 = [
    "강남구","강동구","강북구","강서구","관악구","광진구","구로구","금천구","노원구","도봉구",
    "동대문구","동작구","마포구","서대문구","서초구","성동구","성북구","송파구","양천구","영등포구",
    "용산구","은평구","종로구","중구","중랑구"
]

def make_seoul_wide(df_raw: pd.DataFrame, value_name: str) -> pd.DataFrame:
    # 1) long 변환
    tmp = df_raw.melt(id_vars="지역명", var_name="날짜", value_name=value_name).copy()

    # 2) 타입 정리
    tmp["날짜"] = pd.to_datetime(tmp["날짜"], errors="coerce")
    tmp[value_name] = pd.to_numeric(tmp[value_name].replace("-", np.nan), errors="coerce")

    # 3) 지역명 정리: '서울 ' 접두사가 있으면 제거 (없으면 그대로)
    tmp["지역명"] = tmp["지역명"].astype(str).str.strip()
    tmp["지역명"] = tmp["지역명"].str.replace("서울특별시", "서울", regex=False)
    tmp["지역명"] = tmp["지역명"].str.replace(r"^서울\s*", "", regex=True).str.strip()

    # 4) 서울 25개구만 남기기
    tmp = tmp[tmp["지역명"].isin(SEOUL_GU_25)].copy()

    # 5) wide로 변환
    wide = (
        tmp.pivot_table(index="날짜", columns="지역명", values=value_name, aggfunc="mean")
        .sort_index()
    )

    # 6) 컬럼 순서 고정 + 중복 제거(혹시 생길 경우 대비)
    wide = wide.loc[:, ~wide.columns.duplicated()]
    wide = wide.reindex(columns=SEOUL_GU_25)

    return wide

df_sale_seoul = make_seoul_wide(df_sale, "매매가격지수")
df_rent_seoul = make_seoul_wide(df_rent, "전세가격지수")

print("df_sale_seoul shape:", df_sale_seoul.shape)
print("df_rent_seoul shape:", df_rent_seoul.shape)

display(df_sale_seoul.head(3))
display(df_rent_seoul.head(3))

### (체크) 서울 25개 구가 다 들어왔는지 확인
아래 셀 실행 후:
- 누락된 구가 있는지 확인
- 컬럼 중복이 있었는지 확인


In [ ]:
missing_sale = [g for g in SEOUL_GU_25 if g not in df_sale_seoul.columns]
missing_rent = [g for g in SEOUL_GU_25 if g not in df_rent_seoul.columns]

print("매매 누락 구:", missing_sale)
print("전세 누락 구:", missing_rent)

print("매매 중복 컬럼 수:", df_sale_seoul.columns.duplicated().sum())
print("전세 중복 컬럼 수:", df_rent_seoul.columns.duplicated().sum())

## 4. long 형태로 변환
목표:
- `df_sale_seoul_long`: columns = `지역명`, `매매가격지수` (index=날짜)
- `df_rent_seoul_long`: columns = `지역명`, `전세가격지수` (index=날짜)
- 기대 shape: **(기간수×25, 2)** → 예: 831×25=20775

실습에서는 판다스 버전 이슈를 피하기 위해 `future_stack=True`를 쓰지 않고, 가장 기본 `stack()`을 사용합니다.

In [ ]:
df_sale_seoul_long = (
    df_sale_seoul
    .stack(dropna=False)                         # (날짜, 지역명) -> 값
    .rename("매매가격지수")
    .reset_index()
    .rename(columns={"level_1": "지역명"})
    .set_index("날짜")
    [["지역명", "매매가격지수"]]
)

df_rent_seoul_long = (
    df_rent_seoul
    .stack(dropna=False)
    .rename("전세가격지수")
    .reset_index()
    .rename(columns={"level_1": "지역명"})
    .set_index("날짜")
    [["지역명", "전세가격지수"]]
)

df_sale_seoul_long.shape, df_rent_seoul_long.shape

### (확인) long 데이터 미리보기
아래 셀 실행 후, 날짜 index가 잘 붙었는지 확인하세요.

In [ ]:
display(df_sale_seoul_long.head(10))
display(df_rent_seoul_long.head(10))

## 5. 시각화 1: 강남구 매매/전세 시계열
목표: 강남구의 날짜별 매매·전세 지수를 한 그래프에 그립니다.

In [ ]:
target = "강남구"

plot_df = pd.DataFrame({
    "날짜": df_sale_seoul.index,
    "매매가격지수": df_sale_seoul[target].values,
    "전세가격지수": df_rent_seoul.reindex(df_sale_seoul.index)[target].values
}).dropna()

plt.figure(figsize=(12,5))
sns.lineplot(data=plot_df, x="날짜", y="매매가격지수", label="매매")
sns.lineplot(data=plot_df, x="날짜", y="전세가격지수", label="전세")
plt.title(f"{target} 날짜별 매매/전세 가격지수")
plt.xlabel("날짜"); plt.ylabel("가격지수")
plt.legend()
plt.tight_layout()
plt.show()

plot_df.head()

## 6. 시각화 2: 강남구 매매 vs 전세 상관관계(산점도)
목표: 매매지수와 전세지수의 동행 관계를 산점도로 확인합니다.

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(data=plot_df, x="매매가격지수", y="전세가격지수")
sns.regplot(data=plot_df, x="매매가격지수", y="전세가격지수", scatter=False)
plt.title(f"{target} 매매 vs 전세 상관관계")
plt.tight_layout()
plt.show()

corr = plot_df[["매매가격지수","전세가격지수"]].corr().iloc[0,1]
print("상관계수:", round(float(corr), 4))

## 7. 시각화 3: 매매-전세 격차가 큰/작은 구 찾기
아이디어:
- 같은 날짜에 `매매지수 - 전세지수`를 계산
- 전체 기간 평균 격차가 큰 구/작은 구를 찾기
- 해당 구의 시계열을 그려 비교하기

아래 셀 실행 후, Top/Bottom 구 이름이 출력되는지 확인하세요.

In [ ]:
spread = df_sale_seoul - df_rent_seoul
mean_spread = spread.mean().sort_values(ascending=False)

top_gu = mean_spread.index[0]
bottom_gu = mean_spread.index[-1]

print("평균 격차(매매-전세) 가장 큰 구:", top_gu, "|", round(mean_spread.iloc[0], 3))
print("평균 격차(매매-전세) 가장 작은 구:", bottom_gu, "|", round(mean_spread.iloc[-1], 3))

# 두 구의 시계열 비교
compare_df = pd.DataFrame({
    "날짜": spread.index,
    f"{top_gu} 격차": spread[top_gu].values,
    f"{bottom_gu} 격차": spread[bottom_gu].values
}).dropna()

plt.figure(figsize=(12,5))
sns.lineplot(data=compare_df, x="날짜", y=f"{top_gu} 격차", label=f"{top_gu} 격차")
sns.lineplot(data=compare_df, x="날짜", y=f"{bottom_gu} 격차", label=f"{bottom_gu} 격차")
plt.title("매매-전세 격차 비교")
plt.xlabel("날짜"); plt.ylabel("격차(지수)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. 시각화 4: 연도별 평균 지수(강남구)
목표: 연도별로 평균 매매/전세 지수를 집계해서 막대그래프로 확인합니다.

In [ ]:
tmp = plot_df.copy()
tmp["연도"] = tmp["날짜"].dt.year

yearly = tmp.groupby("연도")[["매매가격지수","전세가격지수"]].mean().reset_index()

display(yearly)

plt.figure(figsize=(10,4))
sns.lineplot(data=yearly, x="연도", y="매매가격지수", marker="o", label="매매")
sns.lineplot(data=yearly, x="연도", y="전세가격지수", marker="o", label="전세")
plt.title(f"{target} 연도별 평균 가격지수")
plt.tight_layout()
plt.show()

## 9. (선택) 지도 시각화(HeatMap) — 전세 변동률
이 파트는 `folium`이 설치되어 있어야 합니다.
실습에서는 **구 이름(district) + 위도(lat) + 경도(lon)** 데이터가 필요합니다.

### 실습 흐름
1) 기준일(첫 날짜) 대비 최신일(마지막 날짜) 전세 변동률 계산
2) 위경도 데이터와 병합
3) HeatMap 표시

⚠️ 만약 folium이 없다면: `pip install folium` 후 커널 재시작이 필요할 수 있어요.

In [ ]:
# (선택) folium이 설치되어 있을 때만 실행
try:
    import folium
    from folium.plugins import HeatMap
except Exception as e:
    print("folium import 실패:", e)
    folium = None

if folium is not None:
    # 1) 전세 변동률(%): 첫 날짜 대비 마지막 날짜
    base_date = df_rent_seoul.index.min()
    target_date = df_rent_seoul.index.max()

    rent_change_pct = (df_rent_seoul.loc[target_date] / df_rent_seoul.loc[base_date] - 1) * 100
    rent_change_pct = rent_change_pct.rename("rent_change_pct").reset_index()
    rent_change_pct.columns = ["district", "rent_change_pct"]

    # 2) 위경도 데이터(예시) — 실습에서 제공된 data 변수가 있으면 그걸 사용
    #    없다면 아래 예시 리스트를 사용하세요.
    if "data" not in globals():
        data = [
            {"district":"강남구","lat":37.5172,"lon":127.0473},
            {"district":"서초구","lat":37.4837,"lon":127.0324},
            {"district":"송파구","lat":37.5146,"lon":127.1059},
            {"district":"영등포구","lat":37.5264,"lon":126.8962},
            # ... (실습에서는 25개 구 전체 좌표를 넣으면 가장 좋습니다)
        ]
        print("data 변수가 없어서 예시 좌표 일부만 사용합니다. (25개 전체를 넣으면 더 정확합니다.)")

    df_coords = pd.DataFrame(data)

    # 3) 병합
    heat_df = df_coords.merge(rent_change_pct, on="district", how="left")

    print("기준일:", base_date.date(), "| 비교일:", target_date.date())
    display(heat_df.head())

    # 4) 지도
    m = folium.Map(location=[37.5665, 126.9780], zoom_start=11)
    heat_data = heat_df[["lat","lon","rent_change_pct"]].dropna().values.tolist()
    HeatMap(heat_data, radius=18, blur=12).add_to(m)

    for _, r in heat_df.dropna(subset=["rent_change_pct"]).iterrows():
        folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=6,
            popup=f'{r["district"]}: {r["rent_change_pct"]:.1f}%',
            fill=True
        ).add_to(m)

    m

## 10. 마무리 체크리스트
- [ ] df_sale_seoul / df_rent_seoul가 비어있지 않다
- [ ] 서울 25개 구 컬럼이 유지된다
- [ ] df_sale_seoul_long / df_rent_seoul_long shape가 `기간×25`로 나온다
- [ ] 강남구 시계열 그래프에 한글이 깨지지 않는다

여기까지 완료되면, 다음 단계로 **수익률(ROE) 계산**이나 **예측**을 붙이는 실습으로 확장할 수 있어요.